<a href="https://colab.research.google.com/github/RuiRodrigues-lab/DataScienceFE/blob/Locker/CP4_7(TH).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.diagnostic import lilliefors
from scipy.stats import wilcoxon
import rpy2.robjects as ro
from scipy.stats import ttest_ind #independente
from scipy.stats import ttest_rel, t #emparelhados
from scipy.stats import mannwhitneyu
from scipy.stats import rankdata

#Precisamos desta biblioteca para podermos escolher um ficheiro local
#Se o ficheiro vier por API ou tivermos um link, é so alterar a forma de import
from google.colab import files

# 1️⃣ Faz upload do ficheiro (vai abrir uma janela para escolher no teu PC)
uploaded = files.upload()

# 2️⃣ Guarda o nome do ficheiro (Colab mostra o nome depois do upload)
filename = list(uploaded.keys())[0]

# 3️⃣ Lê o Excel, por default lê sempre a primeira tab, por isso podemos usar o "sheet_name"
# Se tivermos dados em varias tabs, devemos usar uma Dataframe(df) para cada uma das tabs
df = pd.read_excel(filename, sheet_name='Exerc7')
df.head()

Saving CP4.xlsx to CP4.xlsx


,Vendas,Metodo
0,2,A
1,11,A
2,14,A
3,10,A
4,13,A


In [2]:
n_total = len(df["Vendas"])
n_non_missing = df["Vendas"].notna().sum()

print("Total:", n_total)
print("Não-missing:", n_non_missing)


Total: 28
Não-missing: 28


In [3]:
# 2) Garantir que Metodo é categórico com níveis A e B
df["Metodo"] = pd.Categorical(df["Metodo"], categories=["A", "B"])

# 3) Criar grupos A e B (como grupoA e grupoB no R)
grupoA = df.loc[df["Metodo"] == "A", "Vendas"].dropna()
grupoB = df.loc[df["Metodo"] == "B", "Vendas"].dropna()

print("\nlength(grupoA) =", len(grupoA))
print("length(grupoB) =", len(grupoB))


length(grupoA) = 16
length(grupoB) = 12


In [5]:
# Teste de normalidade (Shapiro-Wilk) (Para testar a normalidade da amostra!)
shapiro_result = stats.shapiro(df["Vendas"].dropna())
print(shapiro_result)

# Teste t unilateral (mean > 0)
t_result = stats.ttest_1samp(df["Vendas"].dropna(), popmean=0, alternative="greater")
print(t_result)

w, p = stats.shapiro(df["Vendas"].dropna())

print("\n Teste de normalidade (Shapiro-Wilk)")
print("\n W =", w)
print("\n p-value =", p)

ShapiroResult(statistic=np.float64(0.8905364331571981), pvalue=np.float64(0.0069164873774635646))
TtestResult(statistic=np.float64(22.661199283684255), pvalue=np.float64(2.0984311528837735e-19), df=np.int64(27))

 Teste de normalidade (Shapiro-Wilk)

 W = 0.8905364331571981

 p-value = 0.0069164873774635646


In [13]:
# Teste de normalidade
W, p = stats.shapiro(grupoA.dropna())

print("Shapiro-Wilk normality test")
print("data: grupoA")
print(f"W = {W:.5f}, p-value = {p:.6f}")

W, p = stats.shapiro(grupoB.dropna())

print("\nShapiro-Wilk normality test")
print("data: grupoB")
print(f"W = {W:.5f}, p-value = {p:.6f}")

print("\nlength(grupoA)")
print(len(grupoA))

print("\nlength(grupoB)")
print(len(grupoB))

print("\nmean_grupoA")
print(grupoA.mean())

print("\nmean_grupoB")
print(grupoB.mean())

Shapiro-Wilk normality test
data: grupoA
W = 0.80424, p-value = 0.003102

Shapiro-Wilk normality test
data: grupoB
W = 0.92527, p-value = 0.332621

length(grupoA)
16

length(grupoB)
12

mean_grupoA
11.5

mean_grupoB
10.083333333333334


In [17]:
# Teste de Wilcoxon-Mann-Whitney (equivalente ao wilcox_test do R)
stat, p_value = mannwhitneyu(grupoA, grupoB, alternative="greater")

print("\nWilcoxon-Mann-Whitney Test")
print("data: Vendas by Metodo (A, B)")
print("U =", stat)
print("p-value =", p_value)

#p-value = 0.010739904642586777 < 0,05 → rejeitar H0
#A função de distribuição dos valores das vendas das lojas que aplicaram o método de
#publicidade A está à direita da função de distribuição dos valores das vendas das lojas que
#aplicaram o método de publicidade B. O método de publicidade A é mais eficaz


Wilcoxon-Mann-Whitney Test
data: Vendas by Metodo (A, B)
U = 145.5
p-value = 0.010739904642586777
